In [13]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import spacy
from spacy.matcher import DependencyMatcher
from spacy.displacy import parse_deps
import spacy.displacy as displacy
import sys
import tqdm
import random

sys.path.append("../src")

from shared import tqdm_readlines

In [2]:
rus_nlp = spacy.load("ru_core_news_md")

In [3]:
rus_snts = dict([(ln.strip().split("\t")[0], rus_nlp(ln.strip().split("\t")[1])) for (ln, _) in zip(tqdm_readlines("../data/tatoeba/rus.tsv"), range(200000))])

 35%|███████████████████▋                                     | 200000/579156 [09:56<18:51, 335.15it/s]


In [30]:
s = rus_nlp("Я ездил на пляж.")
s = rus_nlp("Я был в школе.")
print(parse_deps(s))

{'words': [{'text': 'Я', 'tag': 'PRON', 'lemma': None}, {'text': 'был', 'tag': 'AUX', 'lemma': None}, {'text': 'в', 'tag': 'ADP', 'lemma': None}, {'text': 'школе.', 'tag': 'NOUN', 'lemma': None}], 'arcs': [{'start': 0, 'end': 3, 'label': 'nsubj', 'dir': 'left'}, {'start': 1, 'end': 3, 'label': 'cop', 'dir': 'left'}, {'start': 2, 'end': 3, 'label': 'case', 'dir': 'left'}], 'settings': {'lang': 'ru', 'direction': 'ltr'}}


In [5]:
def search_nlp(nlp, ptn):
    matcher = DependencyMatcher(nlp.vocab)
    matcher.add("PTN", [ptn])
    return matcher

In [42]:
FOOD_DRINK_PTN = [
{
    "RIGHT_ID": "anchor",
    "RIGHT_ATTRS": { 
        "POS": "VERB", 
        # "MORPH": { "IS_SUPERSET": ["Case=Loc"] },
        "LEMMA": { "IN": ["есть", "поесть", "кушать", "жрать", "пить", "выпить"] }
    }
}, 
{
    "LEFT_ID": "anchor", 
    "REL_OP": ">", 
    "RIGHT_ID": "object", 
    "RIGHT_ATTRS": {
        "POS": "NOUN",
        "DEP": "obj"
    }
}]

CLOTHING_PTN = [
{
    "RIGHT_ID": "anchor",
    "RIGHT_ATTRS": { 
        "POS": "VERB", 
        # "MORPH": { "IS_SUPERSET": ["Case=Loc"] },
        "LEMMA": { "IN": ["надевать", "надеть", "примерять", "примерить"] }
    }
}, 
{
    "LEFT_ID": "anchor", 
    "REL_OP": ">", 
    "RIGHT_ID": "object", 
    "RIGHT_ATTRS": {
        "POS": "NOUN",
        "DEP": "obj"
    }
}]

LOCATION_PTN = [
{
    "RIGHT_ID": "anchor",
    "RIGHT_ATTRS": { 
        "POS": "NOUN", 
        # "MORPH": { "IS_SUPERSET": ["Case=Loc"] },
        # "TEXT": { "IN": ["я", "ты", "мы", "вы"] }
    }
}, 
{
    "LEFT_ID": "anchor", 
    "REL_OP": ">", 
    "RIGHT_ID": "object", 
    "RIGHT_ATTRS": {
        "POS": "NOUN",
        "DEP": "nsubj",
        "MORPH": { "IS_SUPERSET": ["Case=Loc"] }
    }
}]

In [43]:
# mtch = search_nlp(rus_nlp, FOOD_DRINK_PTN)
# mtch = search_nlp(rus_nlp, CLOTHING_PTN)
mtch = search_nlp(rus_nlp, LOCATION_PTN)

ws = Counter()
for s, i in tqdm.tqdm(list(zip(rus_snts.values(), range(999999)))):
    m = mtch(s)
    if len(m) > 0:
        # print(i, s, m, s[m[0][1][0]])
        # print(m)
        # print(i, s[m[0][1][1]], "+", s[m[0][1][0]], "|", s)
        w = s[m[0][1][1]].lemma_
        ws[w] += 1
        # print(s[m[0][1][1]].lemma_)

for (lem, c) in ws.most_common():
    print(f"{c}\t{lem}")

100%|███████████████████████████████████████████████████████| 200000/200000 [00:03<00:00, 64131.33it/s]

6	возраст
1	карман
1	шёл
1	язык
1	-
1	платье
